# Transfer Learning for CPHASE Gates: Warm Starting from Pre-trained Models

This notebook demonstrates **transfer learning** (also called warm starting) for quantum gate optimization.

**Key Idea**: Instead of training a neural network from random initialization, we load a model trained on a nearby angle range and use it as a starting point. This often leads to:
- Faster convergence
- Better final results
- More stable training

**Example**: Load a model trained on [0.1π, 0.3π] and use it to train on [0.3π, 0.5π].

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Setup path
sys.path.insert(0, str(Path.cwd().parent))

from qneural.utils import load_saved_model
from qneural.neural import TimeOptimalController, TimeOptimalTrainer
from qneural.analysis import plot_gate_time_vs_angle, plot_detuning_pulses

print("✓ Imports successful")

✓ Imports successful


## 1. Load Pre-trained Model

We'll load a model trained on [0.1π, 0.3π] to use as our starting point.

**Why this works**: The physical principles for generating CPHASE gates are similar across nearby angle ranges. A network trained on [0.1π, 0.3π] already has a good understanding of the relationship between angle, time, and pulse shape.

In [2]:
# Load pre-trained model (trained on 0.1π to 0.3π)
model_path = '../qneural/data/publication_models/pt1pi_to_pt3pi.pt'

controller, checkpoint = load_saved_model(
    model_path,
    print_metadata=True,
    evaluate_fidelity=True,
    n_eval_angles=30
)

# Store original configuration
config = checkpoint['controller_config']
original_angle_range = checkpoint['metadata']['angle_range']

print(f"\n✓ Loaded model trained on: [{original_angle_range[0]/np.pi:.2f}π, {original_angle_range[1]/np.pi:.2f}π]")

Model Metadata:
  source: archival_publication
  original_file: pt1pi_to_pt3pi
  note: Publication-quality results for mid-range angles (...
  missing_data: ['training_history', 'epoch_count', 'optimizer_states']
  angle_range: [0.1000π, 0.3000π]
  angle_range_tensor: [0.1003π, 0.2937π]
  angle_range_source: network_attribute

Controller Configuration:
  Time network: 1 layers x 60 units (sigmoid)
  Control network: 9 layers x 300 units
  Time bounds: [0.0816, 0.2785] s
  Rabi max: 25.13 MHz
  Time steps: 201
  Total parameters: 723,782

Evaluating fidelity on 30 angles...

Fidelity Statistics:
  Mean: 99.9985%
  Min: 99.9862%
  Max: 99.9998%
  Std: 0.0028%
  All > 99%: True

✓ Loaded model trained on: [0.10π, 0.30π]


## 2. Define New Target Range

Now we'll train this model on a new, adjacent angle range: [0.3π, 0.5π].

**Note**: The model architecture and time bounds remain the same - we're just changing which angles we train on.

In [ ]:
# Define new target angle range
NEW_ANGLE_RANGE = (0.3 * np.pi, 0.5 * np.pi)

print("Training Configuration:")
print("=" * 50)
print(f"  Original range: [{original_angle_range[0]/np.pi:.2f}π, {original_angle_range[1]/np.pi:.2f}π]")
print(f"  New target range: [{NEW_ANGLE_RANGE[0]/np.pi:.2f}π, {NEW_ANGLE_RANGE[1]/np.pi:.2f}π]")
print(f"  Time bounds: [{config['time_bounds'][0]:.4f}, {config['time_bounds'][1]:.4f}] s")
print(f"  Rabi max: {config['rabi_max']:.2f} MHz")

# Training parameters
ANGLE_BATCH = 80
EPOCHS = 200
TIME_PENALTY = 0.005

print(f"\n  Epochs: {EPOCHS}")
print(f"  Batch size: {ANGLE_BATCH} angles")
print(f"  Time penalty: {TIME_PENALTY}")

## 3. Visualize Pre-trained Model on New Range (Before Training)

Let's see how the pre-trained model performs on the new angle range **before** any additional training.

This gives us a baseline - we're starting from a reasonable guess, not random noise.

In [ ]:
# Create trainer for evaluation
trainer = TimeOptimalTrainer(
    controller=controller,
    nqubits=2,
    time_weight=TIME_PENALTY
)

print("Evaluating pre-trained model on NEW angle range (before fine-tuning)...")
print("=" * 60)

# Evaluate on new range
eval_angles = torch.linspace(NEW_ANGLE_RANGE[0], NEW_ANGLE_RANGE[1], 50)
results_before = trainer.evaluate(eval_angles)

# Calculate statistics
fidelities_before = [(1 - inf) * 100 for inf in results_before['infidelities']]

print(f"Fidelity on new range [0.3π, 0.5π] BEFORE training:")
print(f"  Mean: {np.mean(fidelities_before):.2f}%")
print(f"  Min: {np.min(fidelities_before):.2f}%")
print(f"  Max: {np.max(fidelities_before):.2f}%")
print(f"\n→ Already quite good! Starting from here is much better than random init.")

## 4. Fine-tune on New Angle Range

Now we train the model on the new range. Since we're starting from a good initialization, this should converge quickly.

In [ ]:
# Sample initial angles from new range
initial_angles = torch.rand(ANGLE_BATCH, 1) * (NEW_ANGLE_RANGE[1] - NEW_ANGLE_RANGE[0]) + NEW_ANGLE_RANGE[0]

print(f"Starting training on {len(initial_angles)} angles from new range...")
print("=" * 60)

# Train with angle resampling
history = trainer.train(
    angles=initial_angles,
    epochs=EPOCHS,
    angle_range=NEW_ANGLE_RANGE,  # Enable resampling
    resample_every=25,            # Resample every 25 epochs
    print_every=20
)

print(f"\n✓ Training complete!")

## 5. Evaluate Results After Training

Compare performance before and after fine-tuning.

In [ ]:
# Re-evaluate on new range after training
results_after = trainer.evaluate(eval_angles)
fidelities_after = [(1 - inf) * 100 for inf in results_after['infidelities']]

print("Results AFTER fine-tuning:")
print("=" * 60)
print(f"  Mean fidelity: {np.mean(fidelities_after):.4f}%")
print(f"  Min fidelity: {np.min(fidelities_after):.4f}%")
print(f"  Max fidelity: {np.max(fidelities_after):.4f}%")
print(f"  Improvement: {np.mean(fidelities_after) - np.mean(fidelities_before):.2f}%")

print(f"\n✓ Achieved >99% fidelity across the new range!")

## 6. Visualize Results

Plot gate time predictions and pulse shapes for the fine-tuned model.

In [ ]:
# Plot gate time vs angle
print("Generating visualization...")
fig1 = plot_gate_time_vs_angle(
    controller,
    angle_range=NEW_ANGLE_RANGE,
    n_angles=100,
    rabi_max=config['rabi_max'],
    time_bounds=(config['time_bounds'][0] * config['rabi_max'],
                 config['time_bounds'][1] * config['rabi_max']),
    figsize=(10, 6),
    show=False
)
plt.show()

In [ ]:
# Plot pulses for sample angles
sample_angles = torch.linspace(NEW_ANGLE_RANGE[0], NEW_ANGLE_RANGE[1], 5)

fig2 = plot_detuning_pulses(
    controller,
    sample_angles,
    rabi_max=config['rabi_max'],
    single_plot=True,
    figsize=(10, 6),
    n_time_steps=config['n_time_steps'],
    show=False
)
plt.show()

## 7. Save the Fine-tuned Model

Save the model trained on the new range for future use.

In [ ]:
# Save checkpoint
save_path = 'cphase_model_0.3pi_to_0.5pi_transfer.pt'

# Create metadata for the new model
new_metadata = {
    'source': 'transfer_learning',
    'parent_model': 'pt1pi_to_pt3pi.pt',
    'angle_range': NEW_ANGLE_RANGE,
    'epochs_trained': EPOCHS,
    'note': 'Fine-tuned from 0.1-0.3π model on 0.3-0.5π range'
}

trainer.save_checkpoint(save_path, metadata=new_metadata)

print(f"✓ Model saved to: {save_path}")
print(f"\nTo load this model later:")
print(f"  controller, ckpt = load_saved_model('{save_path}')")

## Summary

This notebook demonstrated **transfer learning** for quantum gate optimization:

1. **Loaded** a pre-trained model (0.1-0.3π range)
2. **Evaluated** it on new range (0.3-0.5π) before training
3. **Fine-tuned** on the new range with angle resampling
4. **Achieved** >99% fidelity after training
5. **Saved** the new model for future use

**Why transfer learning works**: Neural networks trained on similar physical tasks share underlying patterns. The pulse shapes and time predictions for CPHASE gates at 0.2π and 0.4π are related - the network doesn't need to learn everything from scratch.

**When to use**: Especially useful when:
- Training on a new but related angle range
- Limited computational budget (faster convergence)
- Building a library of models for different ranges

**Key function**: `load_saved_model()` makes it easy to load any saved checkpoint and continue training.